In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



<Accordion>
<AccordionItem title="เวอร์ชันแพ็คเกจ">

โค้ดในหน้านี้ได้รับการพัฒนาโดยใช้ข้อกำหนดต่อไปนี้
แนะนำให้ใช้เวอร์ชันเหล่านี้หรือใหม่กว่า

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
คุณสามารถใช้ตัวเลือกเพื่อปรับแต่ง Estimator primitive ในขณะที่ interface ของเมธอด `run()` ของ primitives นั้นเหมือนกันในทุกการใช้งาน แต่ตัวเลือกของพวกเขาไม่เหมือนกัน ดูข้อมูลเกี่ยวกับตัวเลือก [`qiskit.primitives.BaseEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV2) และ [`qiskit_aer.BaseEstimatorV2`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.primitives.EstimatorV2.html) ได้ที่ API references

<span id="pass-options"></span>

## ตั้งค่าตัวเลือก Estimator

คุณสามารถตั้งค่าตัวเลือกเมื่อ initialize Estimator หลังจาก initialize Estimator หรือคุณสามารถอัปเดตตัวเลือกหลังจาก Estimator ได้รับการ initialize แล้ว สำหรับคำแนะนำในการใช้เทคนิคเหล่านี้ ดู topic [บทนำสู่ตัวเลือก](/guides/runtime-options-overview#options-precedence)

นอกจากนี้ คุณสามารถตั้งค่า `precision` ในเมธอด `run()` ตามที่อธิบายในส่วนต่อไปนี้
<span id="run-method"></span>

### เมธอด run()

ค่าที่คุณสามารถส่งไปยัง `run()` ได้เฉพาะค่าที่กำหนดไว้ใน interface เท่านั้น นั่นคือ `precision` สำหรับ Estimator ซึ่งจะเขียนทับค่าใดก็ตามที่ตั้งไว้สำหรับ `default_precision` ในการรันปัจจุบัน

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime.options import EstimatorOptions

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

options = EstimatorOptions(
    resilience_level=2,
    resilience={"zne_mitigation": True, "zne": {"noise_factors": [1, 3, 5]}},
)

# or...
options = EstimatorOptions()
options.resilience_level = 2
options.resilience.zne_mitigation = True
options.resilience.zne.noise_factors = [1, 3, 5]

estimator = Estimator(mode=backend, options=options)

#### Dictionary

You can specify options as a dictionary when initializing Estimator.

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during initialization
estimator = Estimator(
    backend,
    options={
        "resilience_level": 2,
        "resilience": {
            "zne_mitigation": True,
            "zne": {"noise_factors": [1, 3, 5]},
        },
    },
)

### กรณีพิเศษ: precision
เมธอด `EstimatorV2.run` รับ arguments สองตัว: รายการของ PUBs ซึ่งแต่ละอันสามารถระบุค่าเฉพาะ PUB สำหรับ precision และ keyword argument ของ precision ค่า precision เหล่านี้เป็นส่วนหนึ่งของ interface การรัน Estimator และเป็นอิสระจากตัวเลือกของ Runtime Estimator ค่าเหล่านี้มีความสำคัญเหนือกว่าค่าใดๆ ที่ระบุเป็นตัวเลือก เพื่อให้สอดคล้องกับ Estimator abstraction

อย่างไรก็ตาม หาก `precision` ไม่ได้ถูกระบุโดย PUB หรือใน keyword argument ของ run (หรือหากทั้งหมดเป็น `None`) ค่า precision จากตัวเลือกจะถูกใช้ โดยเฉพาะ `default_precision`

โปรดทราบว่าตัวเลือก Estimator มีทั้ง `default_shots` และ `default_precision` อย่างไรก็ตาม เนื่องจาก gate-twirling เปิดใช้งานโดยค่าเริ่มต้น ผลคูณของ `num_randomizations` และ `shots_per_randomization` จะมีความสำคัญเหนือกว่าตัวเลือกทั้งสองนั้น

โดยเฉพาะสำหรับ Estimator PUB ใดๆ:

1. หาก PUB ระบุ precision ให้ใช้ค่านั้น
2. หาก keyword argument ของ precision ถูกระบุใน `run` ให้ใช้ค่านั้น
3. หาก `twirling` เปิดใช้งาน (True โดยค่าเริ่มต้น) ให้ใช้ผลคูณของ `num_randomizations` และ `shots_per_randomization` ตามที่ระบุใน [ตัวเลือก `twirling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)
4. หาก `estimator.options.default_shots` ถูกระบุ ให้ใช้ค่านั้นเพื่อควบคุมปริมาณข้อมูล
5. หาก `estimator.options.default_precision` ถูกระบุ ให้ใช้ค่านั้น

ตัวอย่างเช่น หาก precision ถูกระบุในสี่ที่ทั้งหมด ค่าที่มีความสำคัญสูงสุด (precision ที่ระบุใน PUB) จะถูกใช้

> **Note:** Precision มีความสัมพันธ์แบบผกผันกับการใช้งาน กล่าวคือ ยิ่ง precision ต่ำ ยิ่งใช้เวลา QPU มากขึ้นในการรัน

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(mode=backend)

# Setting options after initialization
# This uses auto-complete.
estimator.options.default_precision = 0.01
# This does bulk update.
estimator.options.update(
    default_precision=0.02, resilience={"zne_mitigation": True}
)

<span id="run-method"></span>
### Run() method

The only values you can pass to `run()` are those defined in the interface.  That is, `precision` for Estimator. This overwrites any value set for `default_precision` for the current run.

In [16]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

observable = SparsePauliOp("Z" * 3)

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)
isa_observable1 = observable.apply_layout(transpiled1.layout)
isa_observable2 = observable.apply_layout(transpiled2.layout)

estimator = Estimator(mode=backend)
# Default precision to use if not specified in run()
estimator.options.default_precision = 0.01
# Run two circuits, requiring a precision of .02 for both.
estimator.run(
    [(transpiled1, isa_observable1), (transpiled2, isa_observable2)],
    precision=0.02,
)

<RuntimeJobV2('d7amh42k86tc73a1sa20', 'estimator')>

<span id="no-error-mitigation"></span>
## ปิดการ mitigate errors และการกดข่มข้อผิดพลาดทั้งหมด
คุณสามารถปิดการ mitigate errors และการกดข่มข้อผิดพลาดทั้งหมดได้ เช่น หากคุณกำลังทำการวิจัยเกี่ยวกับเทคนิคการ mitigation ของคุณเอง เพื่อให้สำเร็จ ตั้งค่า `resilience_level = 0`

ตัวอย่าง:

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

observable = SparsePauliOp("Z" * 3)

circuit = random_iqp(3)
circuit.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

isa_circuit = pass_manager.run(circuit)
isa_observable = observable.apply_layout(isa_circuit.layout)

# Setting precision during primitive initialization
estimator = Estimator(mode=backend, options={"default_precision": 0.05})

# Run with precision=0.02, overwriting the default.
estimator.run(
    [(isa_circuit, isa_observable1)],
    precision=0.02,
)

<RuntimeJobV2('d8286b00bvlc73d1vn50', 'estimator')>

<span id="options-table"></span>
## ตัวเลือกที่มีอยู่
ตารางต่อไปนี้ระบุตัวเลือกจากเวอร์ชันล่าสุดของ `qiskit-ibm-runtime` หากต้องการดูเวอร์ชันตัวเลือกเก่า ไปที่ [API reference ของ `qiskit-ibm-runtime`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime) และเลือกเวอร์ชันก่อนหน้า

<Accordion>
<AccordionItem title="`default_shots`">

จำนวน shots ทั้งหมดที่จะใช้ต่อ circuit ต่อ configuration

**ตัวเลือก**: จำนวนเต็ม >= 0

**ค่าเริ่มต้น**: None

[เอกสาร API `default_shots`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#default_shots)
  </AccordionItem>

<AccordionItem title="`default_precision`">

precision เริ่มต้นที่จะใช้สำหรับ PUB หรือการเรียก `run()` ที่ไม่ระบุ precision

**ตัวเลือก**: จำนวนทศนิยม > 0

**ค่าเริ่มต้น**: 0.015625 (1 / sqrt(4096))

[เอกสาร API `default_precision`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#default_precision)
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling`">

ควบคุมการตั้งค่า dynamical decoupling error mitigation

[เอกสาร API `dynamical_decoupling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#dynamical_decoupling)
<Accordion>
<AccordionItem title="`dynamical_decoupling.enable`">

**ตัวเลือก**: `True`, `False`

**ค่าเริ่มต้น**: `False`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.extra_slack_distribution`">

**ตัวเลือก**: `middle`, `edges`

**ค่าเริ่มต้น**: `middle`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.scheduling_method`">

ตัวเลือก: `asap`, `alap`
ค่าเริ่มต้น: `alap`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.sequence_type`">

ตัวเลือก: `XX`, `XpXm`, `XY4`
ค่าเริ่มต้น: `XX`
  </AccordionItem>

<AccordionItem title="`dynamical_decoupling.skip_reset_qubits`">

ตัวเลือก: `True`, `False`
ค่าเริ่มต้น: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`environment`">

[เอกสาร API `environment`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#environment)
<Accordion>
<AccordionItem title="`environment.callback`">

ฟังก์ชันที่เรียกได้ซึ่งรับ `Job ID` และ `Job result`

**ตัวเลือก**: None

**ค่าเริ่มต้น**: None
  </AccordionItem>

<AccordionItem title="`environment.job_tags`">

รายการ tags

**ตัวเลือก**: None

**ค่าเริ่มต้น**: None
  </AccordionItem>

<AccordionItem title="`environment.log_level`">

**ตัวเลือก**: DEBUG, INFO, WARNING, ERROR, CRITICAL

**ค่าเริ่มต้น**: WARNING
  </AccordionItem>

<AccordionItem title="`environment.private`">

**ตัวเลือก**: `True`, `False`

**ค่าเริ่มต้น**: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`execution`">

[เอกสาร API `execution`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#execution)
<Accordion>
<AccordionItem title="`execution.init_qubits`">
ว่าจะ reset qubit ไปยัง ground state สำหรับแต่ละ shot หรือไม่

**ตัวเลือก**: `True`, `False`

**ค่าเริ่มต้น**: `True`
  </AccordionItem>

<AccordionItem title="`execution.rep_delay`">
ความล่าช้าระหว่างการวัดและ quantum circuit ต่อมา

**ตัวเลือก**: ค่าในช่วงที่จัดหาโดย `backend.rep_delay_range`

**ค่าเริ่มต้น**: กำหนดโดย `backend.default_rep_delay`
  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`max_execution_time`">
จำกัดระยะเวลาที่งานสามารถรันได้ เป็นวินาที ดูรายละเอียดที่ [คู่มือเวลารันสูงสุด](/guides/max-execution-time)

**ตัวเลือก**: จำนวนเต็มเป็นวินาทีในช่วง [1, 10800]

**ค่าเริ่มต้น**: 10800 (3 ชั่วโมง)
  </AccordionItem>

<AccordionItem title="`resilience`">
ตัวเลือก resilience ขั้นสูงเพื่อปรับแต่งกลยุทธ์ resilience

[เอกสาร API `resilience`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience)
<Accordion>
<AccordionItem title="`resilience.layer_noise_learning`">

ตัวเลือกสำหรับการเรียนรู้ layer noise

[เอกสาร API `resilience.layer_noise_learning`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-layer-noise-learning-options)
  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.layer_pair_depths`">

**ตัวเลือก**: list[int] ของ 2-10 ค่าในช่วง [0, 200]

**ค่าเริ่มต้น**: `(0, 1, 2, 4, 16, 32)`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.max_layers_to_learn`">

**ตัวเลือก**: None, จำนวนเต็ม >= 1

**ค่าเริ่มต้น**: `4`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.num_randomizations`">

**ตัวเลือก**: จำนวนเต็ม >= 1

**ค่าเริ่มต้น**: `32`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_learning.shots_per_randomization`">

**ตัวเลือก**: จำนวนเต็ม >= 1

**ค่าเริ่มต้น**: `128`

  </AccordionItem>

<AccordionItem title="`resilience.layer_noise_model`">

**ตัวเลือก**: `NoiseLearnerResult`, `Sequence[LayerError]`

**ค่าเริ่มต้น**: None

  </AccordionItem>

<AccordionItem title="`resilience.measure_mitigation`">

**ตัวเลือก**: `True`, `False`

**ค่าเริ่มต้น**: `True`

  </AccordionItem>

<AccordionItem title="`resilience.measure_noise_learning`">

ตัวเลือกสำหรับการเรียนรู้ measurement noise

[เอกสาร API `resilience.measure_noise_learning`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-measure-noise-learning-options)
  </AccordionItem>

<AccordionItem title="`resilience.measure_noise_learning.num_randomizations`">

**ตัวเลือก**: จำนวนเต็ม >= 1

**ค่าเริ่มต้น**: `32`

  </AccordionItem>

<AccordionItem title="`resilience.measure_noise_learning.shots_per_randomization`">

**ตัวเลือก**: จำนวนเต็ม, `auto`

**ค่าเริ่มต้น**: `auto`

  </AccordionItem>

<AccordionItem title="`resilience.pec_mitigation`">

**ตัวเลือก**: `True`, `False`

**ค่าเริ่มต้น**: `False`

  </AccordionItem>

<AccordionItem title="`resilience.pec`">

ตัวเลือก probabilistic error cancellation mitigation

[เอกสาร API `resilience.pec`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-pec-options)
  </AccordionItem>

<AccordionItem title="`resilience.pec.max_overhead`">

**ตัวเลือก**: `None`, จำนวนเต็ม >= 1

**ค่าเริ่มต้น**: `100`

  </AccordionItem>

<AccordionItem title="`resilience.pec.noise_gain`">

**ตัวเลือก**: `auto`, จำนวนทศนิยมในช่วง [0, 1]

**ค่าเริ่มต้น**: `auto`

  </AccordionItem>

<AccordionItem title="`resilience.zne_mitigation`">

**ตัวเลือก**: `True`, `False`

**ค่าเริ่มต้น**: `False`

  </AccordionItem>

<AccordionItem title="`resilience.zne`">

[เอกสาร API `resilience.zne`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options)
  </AccordionItem>

<AccordionItem title="`resilience.zne.amplifier`">

**ตัวเลือก**: `gate_folding`, `gate_folding_front`, `gate_folding_back`, `pea`

**ค่าเริ่มต้น**: `gate_folding`

  </AccordionItem>

<AccordionItem title="`resilience.zne.extrapolated_noise_factors`">

**ตัวเลือก**: รายการของจำนวนทศนิยม

**ค่าเริ่มต้น**: `[0, *noise_factors]`

  </AccordionItem>

<AccordionItem title="`resilience.zne.extrapolator`">

**ตัวเลือก**: หนึ่งหรือมากกว่า: `exponential`, `linear`, `double_exponential`, `polynomial_degree_(1 <= k <= 7)`, `fallback`

**ค่าเริ่มต้น**: `(exponential, linear)`

  </AccordionItem>

<AccordionItem title="`resilience.zne.noise_factors`">

**ตัวเลือก**: รายการของจำนวนทศนิยม; แต่ละจำนวนทศนิยม >= 1

**ค่าเริ่มต้น**: `(1, 1.5, 2)` สำหรับ `PEA` และ `(1, 3, 5)` สำหรับกรณีอื่น

  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`resilience_level`">

ระดับ resilience ต่อ errors ระดับที่สูงกว่าจะสร้างผลลัพธ์ที่แม่นยำกว่าโดยแลกกับเวลาประมวลผลที่นานกว่า ดูส่วน [ระดับ resilience](/guides/estimator-noise-management#resilience) ใน topic Noise management เพื่อเรียนรู้เพิ่มเติม

**ตัวเลือก**: `0`, `1`, `2`

**ค่าเริ่มต้น**: `1`

[เอกสาร API `resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
  </AccordionItem>

<AccordionItem title="`seed_estimator`">

**ตัวเลือก**: จำนวนเต็ม

**ค่าเริ่มต้น**: None

[`seed_estimator`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#seed_estimator)
  </AccordionItem>

<AccordionItem title="`simulator`">

ตัวเลือกที่จะส่งเมื่อ simulate backend

[เอกสาร API `simulator`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-simulator-options)
<Accordion>
<AccordionItem title="`simulator.basis_gates`">

**ตัวเลือก**: รายการชื่อ basis gate ที่จะ unroll ไปยัง

**ค่าเริ่มต้น**: ชุดของ basis gates ทั้งหมดที่รองรับโดย [Qiskit Aer simulator](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator.html)

  </AccordionItem>

<AccordionItem title="`simulator.coupling_map`">

**ตัวเลือก**: รายการของ directed two-qubit interactions

**ค่าเริ่มต้น**: None ซึ่งหมายถึงไม่มีข้อจำกัดการเชื่อมต่อ (การเชื่อมต่อเต็มรูปแบบ)

  </AccordionItem>

<AccordionItem title="`simulator.noise_model`">

**ตัวเลือก**: [Qiskit Aer NoiseModel](/guides/build-noise-models) หรือการแสดงผล

**ค่าเริ่มต้น**: None

  </AccordionItem>

<AccordionItem title="`simulator.seed_simulator`">

**ตัวเลือก**: จำนวนเต็ม

**ค่าเริ่มต้น**: None

  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`twirling`">

ตัวเลือก Twirling

[เอกสาร API `twirling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)
<Accordion>
<AccordionItem title="`twirling.enable_gates`">

**ตัวเลือก**: True, False

**ค่าเริ่มต้น**: False

  </AccordionItem>

<AccordionItem title="`twirling.enable_measure`">

**ตัวเลือก**: True, False

**ค่าเริ่มต้น**: True

  </AccordionItem>

<AccordionItem title="`twirling.num_randomizations`">

**ตัวเลือก**: `auto`, จำนวนเต็ม >= 1

**ค่าเริ่มต้น**: `auto`

  </AccordionItem>

<AccordionItem title="`twirling.shots_per_randomization`">

**ตัวเลือก**: `auto`, จำนวนเต็ม >= 1

**ค่าเริ่มต้น**: `auto`

  </AccordionItem>

<AccordionItem title="`twirling.strategy`">

**ตัวเลือก**: `active`, `active-circuit`, `active-accum`, `all`

**ค่าเริ่มต้น**: `active-accum`

  </AccordionItem>

</Accordion>

  </AccordionItem>

<AccordionItem title="`experimental`">

ตัวเลือกเชิงทดลอง เมื่อมีให้ใช้งาน

  </AccordionItem>

</Accordion>
## ความเข้ากันได้ของฟีเจอร์
ฟีเจอร์ runtime บางอย่างไม่สามารถใช้ร่วมกันในงานเดียว คลิกแท็บที่เหมาะสมเพื่อดูรายการฟีเจอร์ที่เข้ากันไม่ได้กับฟีเจอร์ที่เลือก:

<Tabs>

  <TabItem value="Fractional" label="Fractional gates">
  เข้ากันไม่ได้กับ:
  - Gate twirling
  - PEA
  - PEC

  </TabItem>

  <TabItem value="ZNE" label="Gate-folding ZNE">
    เข้ากันไม่ได้กับ:
  - PEA
  - PEC

  อาจไม่ทำงานเมื่อใช้ custom gates
  </TabItem>
  <TabItem value="Twirling" label="Gate twirling">
  เข้ากันไม่ได้กับ fractional gates หรือ stretches

  หมายเหตุอื่นๆ:
  - Measurement twirling สามารถใช้ได้กับ terminal measurements เท่านั้น
  - ไม่ทำงานกับ non-Clifford entanglers

  </TabItem>

  <TabItem value="PEA" label="PEA">
    เข้ากันไม่ได้กับ:
  - Fractional gates
  - Gate-folding ZNE
  - PEC
  </TabItem>

  <TabItem value="PEC" label="PEC">
    เข้ากันไม่ได้กับ:
  - Fractional gates
  - Gate-folding ZNE
  - PEA
  </TabItem>

</Tabs>
## ขั้นตอนถัดไป
> **Tip:** - ทบทวนคู่มือ [บทนำสู่ตัวเลือก](/guides/runtime-options-overview)
> - ดูรายละเอียดเพิ่มเติมเกี่ยวกับเมธอด `EstimatorV2` ใน [Estimator API reference](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2)
> - ตัดสินใจว่าจะรันงานใน [execution mode](/guides/execution-modes) ใด
> - เรียนรู้เกี่ยวกับ [การจัดการ noise ด้วย Estimator](/guides/estimator-noise-management)

In [3]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator, QiskitRuntimeService

# Define the service.  This allows you to access an IBM QPU.
service = QiskitRuntimeService()

# Get a backend
backend = service.least_busy(operational=True, simulator=False)

# Define Estimator
estimator = Estimator(backend)

options = estimator.options

# Turn off all error mitigation and suppression
options.resilience_level = 0

<span id="options-table"></span>
## Available options

The following table documents options from the latest version of `qiskit-ibm-runtime`. To see older option versions, visit the [`qiskit-ibm-runtime` API reference](/docs/api/qiskit-ibm-runtime) and select a previous version.

<Accordion>
<AccordionItem title="`default_shots`">

The total number of shots to use per circuit per configuration.

**Choices**: Integer >= 0

**Default**: None

[`default_shots` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#default_shots)
  </AccordionItem>


<AccordionItem title="`default_precision`">

The default precision to use for any PUB or `run()` call that does not specify one.

**Choices**: Float > 0

**Default**: 0.015625 (1 / sqrt(4096))

[`default_precision` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#default_precision)
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling`">

Control dynamical decoupling error mitigation settings.

[`dynamical_decoupling` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#dynamical_decoupling)
<Accordion>
<AccordionItem title="`dynamical_decoupling.enable`">

**Choices**: `True`, `False`

**Default**: `False`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.extra_slack_distribution`">

**Choices**: `middle`, `edges`

**Default**: `middle`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.scheduling_method`">

Choices: `asap`, `alap`
Default: `alap`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.sequence_type`">

Choices: `XX`, `XpXm`, `XY4`
Default: `XX`
  </AccordionItem>


<AccordionItem title="`dynamical_decoupling.skip_reset_qubits`">

Choices: `True`, `False`
Default: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>



<AccordionItem title="`environment`">

[`environment` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#environment)
<Accordion>
<AccordionItem title="`environment.callback`">

Callable function that receives the `Job ID` and `Job result`.

**Choices**: None

**Default**: None
  </AccordionItem>


<AccordionItem title="`environment.job_tags`">

List of tags.

**Choices**: None

**Default**: None
  </AccordionItem>


<AccordionItem title="`environment.log_level`">

**Choices**: DEBUG, INFO, WARNING, ERROR, CRITICAL

**Default**: WARNING
  </AccordionItem>


<AccordionItem title="`environment.private`">

**Choices**: `True`, `False`

**Default**: `False`
  </AccordionItem>

</Accordion>

  </AccordionItem>



<AccordionItem title="`execution`">

[`execution` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#execution)
<Accordion>
<AccordionItem title="`execution.init_qubits`">
Whether to reset the qubits to the ground state for each shot.

**Choices**: `True`, `False`

**Default**: `True`
  </AccordionItem>


<AccordionItem title="`execution.rep_delay`">
The delay between a measurement and the subsequent quantum circuit.

**Choices**: Value in the range supplied by `backend.rep_delay_range`

**Default**: Given by `backend.default_rep_delay`
  </AccordionItem>

</Accordion>

  </AccordionItem>



<AccordionItem title="`max_execution_time`">
Limits how long a job can run, in seconds. See the [maximum execution time](/docs/guides/max-execution-time) guide for details.

**Choices**: Integer number of seconds in the range [1, 10800]

**Default**: 10800 (3 hours)
  </AccordionItem>


<AccordionItem title="`resilience`">
Advanced resilience options to fine tune the resilience strategy.

[`resilience` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#resilience)
<Accordion>
<AccordionItem title="`resilience.layer_noise_learning`">

Options for learning layer noise.

[`resilience.layer_noise_learning` API documentation](/docs/api/qiskit-ibm-runtime/options-layer-noise-learning-options)
  </AccordionItem>


<AccordionItem title="`resilience.layer_noise_learning.layer_pair_depths`">

**Choices**: list[int] of 2-10 values in the range [0, 200]

**Default**: `(0, 1, 2, 4, 16, 32)`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_learning.max_layers_to_learn`">

**Choices**: None, Integer >= 1

**Default**: `4`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_learning.num_randomizations`">

**Choices**: Integer >= 1

**Default**: `32`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_learning.shots_per_randomization`">

**Choices**: Integer >= 1

**Default**: `128`

  </AccordionItem>



<AccordionItem title="`resilience.layer_noise_model`">

**Choices**: `NoiseLearnerResult`, `Sequence[LayerError]`

**Default**: None

  </AccordionItem>



<AccordionItem title="`resilience.measure_mitigation`">

**Choices**: `True`, `False`

**Default**: `True`

  </AccordionItem>



<AccordionItem title="`resilience.measure_noise_learning`">

Options for measurement noise learning.

[`resilience.measure_noise_learning` API documentation](/docs/api/qiskit-ibm-runtime/options-measure-noise-learning-options)
  </AccordionItem>


<AccordionItem title="`resilience.measure_noise_learning.num_randomizations`">

**Choices**: Integer >= 1

**Default**: `32`

  </AccordionItem>



<AccordionItem title="`resilience.measure_noise_learning.shots_per_randomization`">

**Choices**: Integer, `auto`

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`resilience.pec_mitigation`">

**Choices**: `True`, `False`

**Default**: `False`

  </AccordionItem>



<AccordionItem title="`resilience.pec`">

Probabilistic error cancellation mitigation options.

[`resilience.pec` API documentation](/docs/api/qiskit-ibm-runtime/options-pec-options)
  </AccordionItem>


<AccordionItem title="`resilience.pec.max_overhead`">

**Choices**: `None`, Integer >= 1


**Default**: `100`

  </AccordionItem>



<AccordionItem title="`resilience.pec.noise_gain`">

**Choices**: `auto`, float in the range [0, 1]

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`resilience.zne_mitigation`">

**Choices**: `True`, `False`

**Default**: `False`

  </AccordionItem>



<AccordionItem title="`resilience.zne`">

[`resilience.zne` API documentation](/docs/api/qiskit-ibm-runtime/options-zne-options)
  </AccordionItem>


<AccordionItem title="`resilience.zne.amplifier`">

**Choices**: `gate_folding`, `gate_folding_front`, `gate_folding_back`, `pea`

**Default**: `gate_folding`

  </AccordionItem>



<AccordionItem title="`resilience.zne.extrapolated_noise_factors`">

**Choices**: List of floats

**Default**: `[0, *noise_factors]`

  </AccordionItem>



<AccordionItem title="`resilience.zne.extrapolator`">

**Choices**: One or more of: `exponential`, `linear`, `double_exponential`, `polynomial_degree_(1 <= k <= 7)`, `fallback`

**Default**: `(exponential, linear)`

  </AccordionItem>



<AccordionItem title="`resilience.zne.noise_factors`">

**Choices**: List of floats; each float >= 1

**Default**: `(1, 1.5, 2)` for `PEA`, and `(1, 3, 5)` otherwise

  </AccordionItem>


</Accordion>

  </AccordionItem>



<AccordionItem title="`resilience_level`">

How much resilience to build against errors. Higher levels generate more accurate results at the expense of longer processing times. See the [resilience levels](/docs/guides/estimator-noise-management#resilience) section in the Noise management topic to learn more.

**Choices**: `0`, `1`, `2`

**Default**: `1`

[`resilience_level` API documentation](/docs/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
  </AccordionItem>


<AccordionItem title="`seed_estimator`">

**Choices**: Integer

**Default**: None

[`seed_estimator`](/docs/api/qiskit-ibm-runtime/options-estimator-options#seed_estimator)
  </AccordionItem>


<AccordionItem title="`simulator`">

Options to pass when simulating a backend

[`simulator` API documentation](/docs/api/qiskit-ibm-runtime/options-simulator-options)
<Accordion>
<AccordionItem title="`simulator.basis_gates`">

**Choices**: List of basis gate names to unroll to

**Default**: The set of all basis gates supported by [Qiskit Aer simulator](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator.html)

  </AccordionItem>



<AccordionItem title="`simulator.coupling_map`">

**Choices**: List of directed two-qubit interactions

**Default**: None, which implies no connectivity constraints (full connectivity).

  </AccordionItem>



<AccordionItem title="`simulator.noise_model`">

**Choices**: [Qiskit Aer NoiseModel](/docs/guides/build-noise-models), or its representation

**Default**: None

  </AccordionItem>



<AccordionItem title="`simulator.seed_simulator`">

**Choices**: Integer

**Default**: None

  </AccordionItem>


</Accordion>

  </AccordionItem>



<AccordionItem title="`twirling`">

Twirling options

[`twirling` API documentation](/docs/api/qiskit-ibm-runtime/options-twirling-options)
<Accordion>
<AccordionItem title="`twirling.enable_gates`">

**Choices**: True, False

**Default**: False

  </AccordionItem>



<AccordionItem title="`twirling.enable_measure`">

**Choices**: True, False

**Default**: True

  </AccordionItem>



<AccordionItem title="`twirling.num_randomizations`">

**Choices**: `auto`, Integer >= 1

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`twirling.shots_per_randomization`">

**Choices**: `auto`, Integer >= 1

**Default**: `auto`

  </AccordionItem>



<AccordionItem title="`twirling.strategy`">

**Choices**: `active`, `active-circuit`, `active-accum`, `all`

**Default**: `active-accum`

  </AccordionItem>


</Accordion>

  </AccordionItem>



<AccordionItem title="`experimental`">

Experimental options, when available.

  </AccordionItem>


</Accordion>

<span id="options-compatibility-table"></span>
## Feature compatibility

Certain runtime features cannot be used together in a single job. Click the appropriate tab for a list of features that are incompatible with the selected feature:

<Accordion>
  <AccordionItem title="Fractional gates">
    Incompatible with:
  - Gate twirling
  - PEA
  - PEC
  </AccordionItem>
  <AccordionItem title="Gate-folding ZNE">
    Might not work when using custom gates. Incompatible with:
  - PEA
  - PEC
  </AccordionItem>
  <AccordionItem title="Gate twirling">
    Incompatible with:
  - Fractional gates
  - Stretches

  Other notes:
  - Measurement twirling can only be applied to terminal measurements.
  - Does not work with non-Clifford entanglers.
  </AccordionItem>
  <AccordionItem title="PEA">
    Incompatible with:
  - Fractional gates
  - Gate-folding ZNE
  - PEC
  </AccordionItem>
  <AccordionItem title="PEC">
    Incompatible with:
  - Fractional gates
  - Gate-folding ZNE
  - PEA
  </AccordionItem>
</Accordion>

## Next steps

<Admonition type="tip" title="Recommendations">
    - Find more details about the `EstimatorV2` methods in the [Estimator API reference](/docs/api/qiskit-ibm-runtime/estimator-v2).
    - Decide what [execution mode](/docs/guides/execution-modes) to run your job in.
    - Learn about [noise management with Estimator](/docs/guides/estimator-noise-management).
</Admonition>